# NB09 — Ground-Truth Synthetic Deduplication Benchmark

**Follow-up to NB08.** NB08 attempted to validate the specimen-construction technique
against deep-embedding cosine similarity directly on the real 11,094-image dataset, but
that attempt was inconclusive: BDLitchi has no ground-truth leaf-identity labels, so
there was no way to tell a *correct* flag from a *false positive*. The result (87.8% of
cross-specimen pairs flagged) turned out to be a measurement artifact of generic
ImageNet embeddings reacting to BDLitchi's standardised photography protocol, not a
real duplicate-detection signal — see `bdlitchi_status.md` for the full writeup of why
that number was not used.

This notebook supplies the missing ingredient: **ground truth.** It builds a small
synthetic benchmark of 100 images from real BDLitchi photographs, with duplicate
identity assigned by construction rather than inferred by an algorithm. That makes it
possible to compute real precision and recall for every method, including the
deep-embedding approach, instead of interpreting a raw percentage with no reference
point.

### Folder layout

The notebook creates two sub-folders under `synthetic_images/`:

* **`originals/`** — every real photo used (79 images): 58 true singletons plus the 21
  source images that go on to have a synthetic duplicate made from them.
* **`duplicates/`** — the 21 synthetic "second photos" generated from those 21 sources.

`ground_truth.csv` is the answer key: every image's `true_leaf_id` (which physical leaf
it is), `is_derived` (original vs. duplicate), and `variant_type`.

### What "duplicate" means here — three types, 21 pairs total

* **7 exact-copy duplicates** — the original file's bytes, copied with no re-encoding at
  all. This is the only case exact-hash (MD5) matching can ever catch by construction,
  and it exists specifically to give that baseline a fair chance.
* **7 near-exact / burst-style duplicates** — the same frame, aggressively
  re-compressed and lightly resized, mimicking a burst-mode duplicate shot a moment
  later (not byte-identical, but visually near-identical).
* **7 realistic multi-angle duplicates** — rotated, cropped, and
  brightness/contrast-jittered, mimicking a genuinely second photograph of the same leaf
  from a different angle a few seconds to a couple of minutes later. This is exactly the
  case the manuscript's Section 3.4 says pHash alone can miss.

That gives 100 images total (58 + 21 + 21), with exactly 21 known true-duplicate pairs
(and 4,929 known true-negative pairs) — enough to score every method honestly.

### Methods compared

1. **Exact-hash matching (MD5)** — the crudest possible baseline; a floor, not a
   contender.
2. **pHash clustering alone** (Hamming ≤ 5, per class) — the approach used before this
   revision, and the `cluster_id` column already in the manifest.
3. **Full specimen method** (pHash clusters **union** bounded 30 s temporal blocks) —
   this paper's actual technique, `specimen_id`. Code for both 2 and 3 is copied
   verbatim from `nb01-identity-disjoint-splits.ipynb` so this is a faithful replay of
   the real algorithm, not a re-implementation that might silently drift from it.
4. **Deep-embedding cosine similarity** (ImageNet-pretrained ResNet-50, not
   fine-tuned) — the alternative approach evaluated directly on the real corpus in
   NB08. Here, with ground truth available, its ROC-AUC, precision, and recall can be
   reported directly instead of an uninterpretable raw percentage.

### Two scoring methods, and why there are two

The first pass scores every method **pairwise**: precision/recall against all 4,950
possible image pairs. That is the standard way to score a duplicate-finder, but it is
the wrong lens for a method whose real job is train/test leakage prevention, not
duplicate-finding. Pairwise precision punishes an over-merge mistake once for *every
pair* inside the affected group — one accidental merge in a group of 5 images counts as
10 "false positives," even though it is really one lost independent unit, not ten. That
combinatorial inflation hits the full specimen method hardest, since it is the method
most willing to merge (deliberately, by design — see the manuscript's Section 3.4).

The second, headline pass scores every method at the **specimen level** instead:

* **Leakage-prevention recall** — the fraction of true duplicate pairs placed in the
  same group. This is the number that actually matters for split integrity, and it is
  mathematically identical to the pairwise recall above (the notebook cross-checks the
  two agree exactly).
* **Over-merge rate** — how many of the 79 real leaves ended up grouped with a
  genuinely different leaf, counted once per leaf, not once per pair. This is the fair
  way to price the cost of over-merging, and it uses the same units (specimen counts)
  as Table 4 already in the manuscript.

`table_dedup_purity.tex` and `fig_dedup_recall_vs_overmerge.png` are the headline
outputs; the pairwise table and PR-curve figure are kept as supplementary/reference
material only.

### Outputs

The complete `revision_dedup_benchmark` output folder is the deliverable. The headline
files are `table_dedup_purity.tex`, `specimen_purity_metrics.csv`,
`cluster_assignments.csv`, and `fig_dedup_recall_vs_overmerge.png`; everything else
(pairwise table, PR curve, recall by duplicate type, time-gap breakdown, contact sheet,
`manuscript_numbers_dedup_benchmark.md`) is supporting detail.


In [ ]:
# ===== Imports =====
import os, json, time, random, hashlib, warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt

import imagehash
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# ===== Config =====
RAW_DATASET_DIR = '/kaggle/input/datasets/maruf170102/bdlithi/Dataset'   # <-- EDIT: folder containing the 11 class sub-folders

OUT_DIR = Path('/kaggle/working/revision_dedup_benchmark'); OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
SYN_DIR = OUT_DIR / 'synthetic_images'
ORIG_DIR = SYN_DIR / 'originals'; ORIG_DIR.mkdir(parents=True, exist_ok=True)
DUP_DIR = SYN_DIR / 'duplicates'; DUP_DIR.mkdir(parents=True, exist_ok=True)

# ---- Synthetic benchmark composition ----
N_SINGLETON = 58          # real images used as-is, no duplicate created -> originals/ only
N_EXACT_COPY = 7          # true byte-identical duplicates (literal file copy, no re-encoding)
N_NEAR_EXACT = 7          # near-exact / burst-style duplicates (re-compressed + lightly resized)
N_REALISTIC = 7           # realistic multi-angle duplicates (rotate/crop/brightness/blur)
N_DUP_SOURCES = N_EXACT_COPY + N_NEAR_EXACT + N_REALISTIC   # 21 real images that each get ONE duplicate
N_TOTAL_IMAGES = N_SINGLETON + N_DUP_SOURCES + N_DUP_SOURCES  # 58 + 21 + 21 = 100

# originals/ ends up with N_SINGLETON + N_DUP_SOURCES = 79 images (every real photo used);
# duplicates/ ends up with N_DUP_SOURCES = 21 images (one synthetic "second photo" each).

# ---- Synthetic capture-session structure ----
# A real BDLitchi "session" is a whole continuous shooting period covering MANY different
# physical leaves (mean ~427 images/session), not one leaf per session: the photographer
# shoots a short burst of one leaf (~12 s between frames), then moves on to the next leaf
# with a somewhat longer gap, all still inside the same session. We reproduce that
# structure here: each "visit" below is one physical leaf (a singleton image, or a
# duplicate pair), and visits are chained together into sessions with a between-visit gap
# that is *usually* well outside the 30 s block window but occasionally, by chance, falls
# inside it -- exactly the real risk the temporal-block step has to withstand, rather than
# either an easy strawman (visits always far apart) or a tautology (each visit its own
# session, guaranteeing a perfect score with no work done).
N_SESSIONS = 10
BETWEEN_VISIT_GAP_S = (20, 120)    # gap between one leaf-visit ending and the next starting
DUP_TIME_GAP_RANGE_S = (2, 90)     # gap between an original and its OWN duplicate variant --
                                    # deliberately spans across AND beyond the 30 s block
                                    # boundary used in the paper, so we can see how many
                                    # real duplicates a 30 s window actually catches.

# ---- Near-exact / burst-style variant knobs (should look like a re-saved duplicate frame) ----
NEAR_EXACT_JPEG_QUALITY = (30, 55)
NEAR_EXACT_RESIZE_FRAC = (0.92, 0.98)

# ---- Realistic multi-angle variant knobs (should look like a second, deliberate photo) ----
REALISTIC_ROTATE_DEG = (8, 20)
REALISTIC_CROP_FRAC = (0.82, 0.94)
REALISTIC_BLUR_RADIUS = (0.6, 1.3)
REALISTIC_BRIGHTNESS = (0.85, 1.15)
REALISTIC_CONTRAST = (0.85, 1.15)

# ---- Method parameters (must match the manuscript / NB01 exactly) ----
PHASH_THRESHOLD = 5        # Hamming distance <= this -> near-duplicate edge (Section 3.4)
SPECIMEN_BLOCK_SECONDS = 30  # bounded temporal block span used throughout the paper

print('RAW_DATASET_DIR exists:', os.path.isdir(RAW_DATASET_DIR))
print(f'Synthetic set: {N_SINGLETON} singleton + {N_DUP_SOURCES} duplicated-source images '
      f'-> originals/ ({N_SINGLETON + N_DUP_SOURCES} images); '
      f'{N_DUP_SOURCES} duplicate variants -> duplicates/ '
      f'({N_EXACT_COPY} exact-copy, {N_NEAR_EXACT} near-exact, {N_REALISTIC} realistic). '
      f'{N_TOTAL_IMAGES} images total, grouped into {N_SESSIONS} synthetic multi-leaf sessions.')


In [ ]:
# ===== Path verification =====
assert os.path.isdir(RAW_DATASET_DIR), f'RAW_DATASET_DIR not found: {RAW_DATASET_DIR}. Fix the config cell.'
classes = sorted([d.name for d in Path(RAW_DATASET_DIR).iterdir() if d.is_dir()])
all_real_paths = []
for c in classes:
    for p in sorted((Path(RAW_DATASET_DIR) / c).glob('*')):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp', '.webp'):
            all_real_paths.append({'path': str(p), 'label': c})
real_pool = pd.DataFrame(all_real_paths)
assert len(real_pool) >= N_TOTAL_IMAGES - N_DUP_SOURCES, (
    f'Need at least {N_TOTAL_IMAGES - N_DUP_SOURCES} source images, found {len(real_pool)}.')
print(f'OK - {len(classes)} classes, {len(real_pool)} candidate source images found.')


In [ ]:
# ===== Sample real source images and assign ground truth =====
rng = np.random.default_rng(SEED)

n_sources = N_SINGLETON + N_DUP_SOURCES  # 79 distinct real photos = 79 "leaf visits"
sampled = real_pool.sample(n=n_sources, random_state=SEED).reset_index(drop=True)
sampled['true_leaf_id'] = np.arange(n_sources)

# First N_DUP_SOURCES visits (already in random sample order) get a synthetic duplicate.
# Of those: first N_EXACT_COPY -> literal byte-identical copy; next N_NEAR_EXACT ->
# near-exact/burst variant; the rest -> realistic multi-angle variant.
dup_source_mask = np.zeros(n_sources, dtype=bool)
dup_source_mask[:N_DUP_SOURCES] = True
variant_type_for_source = np.array(['none'] * n_sources, dtype=object)
variant_type_for_source[:N_EXACT_COPY] = 'exact_copy'
variant_type_for_source[N_EXACT_COPY:N_EXACT_COPY + N_NEAR_EXACT] = 'near_exact'
variant_type_for_source[N_EXACT_COPY + N_NEAR_EXACT:N_DUP_SOURCES] = 'realistic'
sampled['has_duplicate'] = dup_source_mask
sampled['variant_type'] = variant_type_for_source  # 'none' for true singletons

# ---- Chain the 79 visits into N_SESSIONS sessions, each a sequence of leaf-visits ----
# (This is what makes the session structure realistic: many different leaves per session,
# not one leaf per session -- see the config cell for why.)
visit_order = rng.permutation(n_sources)
session_of_visit = np.array_split(visit_order, N_SESSIONS)
visit_to_session = {}
for s_idx, visits in enumerate(session_of_visit):
    for v in visits:
        visit_to_session[v] = f'syn_sess{s_idx:02d}'
sampled['session_id'] = sampled['true_leaf_id'].map(visit_to_session)

# ---- Walk each session in visit order, laying down timestamps visit by visit ----
session_start_epoch = {sid: 1_700_000_000 + i * 86_400 for i, sid in enumerate(sorted(set(visit_to_session.values())))}
timestamps = np.zeros(n_sources)
dup_variant_gap = {}  # true_leaf_id -> gap (s) to its own duplicate variant
for s_idx, visits in enumerate(session_of_visit):
    sid = f'syn_sess{s_idx:02d}'
    cursor = session_start_epoch[sid]
    for v in visits:
        row = sampled.loc[sampled.true_leaf_id == v].iloc[0]
        timestamps[v] = cursor
        if row.has_duplicate:
            gap = int(rng.integers(DUP_TIME_GAP_RANGE_S[0], DUP_TIME_GAP_RANGE_S[1] + 1))
            dup_variant_gap[v] = gap
            visit_end = cursor + gap
        else:
            visit_end = cursor
        cursor = visit_end + rng.integers(BETWEEN_VISIT_GAP_S[0], BETWEEN_VISIT_GAP_S[1] + 1)
sampled['timestamp'] = timestamps

# ---- Build the N_DUP_SOURCES duplicate-variant rows: same session, offset timestamp ----
dup_rows = []
for _, r in sampled[sampled.has_duplicate].iterrows():
    gap = dup_variant_gap[r.true_leaf_id]
    dup_rows.append({
        'true_leaf_id': r.true_leaf_id, 'label': r.label, 'session_id': r.session_id,
        'timestamp': r.timestamp + gap, 'time_gap_from_source_s': int(gap),
        'variant_type': r.variant_type, 'is_derived': True, 'source_path': r.path,
    })
dup_df = pd.DataFrame(dup_rows)

sampled['is_derived'] = False
sampled['time_gap_from_source_s'] = np.nan
sampled = sampled.rename(columns={'path': 'source_path'})
# Non-derived rows keep their real variant_type ('exact_copy'/'near_exact'/'realistic'/'none')
# so image generation (next cell) knows which originals must be byte-identical to their pair.

gt = pd.concat([
    sampled[['true_leaf_id', 'label', 'session_id', 'timestamp', 'time_gap_from_source_s',
             'variant_type', 'is_derived', 'source_path']],
    dup_df[['true_leaf_id', 'label', 'session_id', 'timestamp', 'time_gap_from_source_s',
            'variant_type', 'is_derived', 'source_path']],
], ignore_index=True)
gt['image_id'] = [f'img{i:03d}' for i in range(len(gt))]

assert len(gt) == N_TOTAL_IMAGES, f'expected {N_TOTAL_IMAGES} images, built {len(gt)}'
n_true_pairs = int((gt.is_derived).sum())
print(f'Ground truth built: {len(gt)} images across {gt.session_id.nunique()} sessions, '
      f'{n_true_pairs} true duplicate pairs.')
print(gt[gt.is_derived].variant_type.value_counts())
print('Visits per session:', sampled.groupby('session_id').size().to_dict())


In [ ]:
# ===== Materialise the synthetic image set on disk: originals/ and duplicates/ =====
import shutil

def make_near_exact(img, rng_local):
    frac = rng_local.uniform(*NEAR_EXACT_RESIZE_FRAC)
    w, h = img.size
    img = img.resize((max(1, int(w * frac)), max(1, int(h * frac))), Image.BILINEAR)
    return img, {'quality': int(rng_local.integers(NEAR_EXACT_JPEG_QUALITY[0], NEAR_EXACT_JPEG_QUALITY[1] + 1))}

def make_realistic(img, rng_local):
    angle = rng_local.uniform(*REALISTIC_ROTATE_DEG) * (1 if rng_local.random() < 0.5 else -1)
    img = img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=(255, 255, 255))
    crop_frac = rng_local.uniform(*REALISTIC_CROP_FRAC)
    w, h = img.size
    cw, ch = int(w * crop_frac), int(h * crop_frac)
    left = rng_local.integers(0, max(1, w - cw + 1))
    top = rng_local.integers(0, max(1, h - ch + 1))
    img = img.crop((left, top, left + cw, top + ch)).resize((w, h), Image.BILINEAR)
    img = ImageEnhance.Brightness(img).enhance(rng_local.uniform(*REALISTIC_BRIGHTNESS))
    img = ImageEnhance.Contrast(img).enhance(rng_local.uniform(*REALISTIC_CONTRAST))
    img = img.filter(ImageFilter.GaussianBlur(radius=rng_local.uniform(*REALISTIC_BLUR_RADIUS)))
    return img, {'quality': 90}

file_paths = []
img_rng = np.random.default_rng(SEED + 1)
for _, r in gt.iterrows():
    dest_dir = DUP_DIR if r.is_derived else ORIG_DIR
    if r.variant_type == 'exact_copy':
        # True byte-identical duplicate: copy the raw file, no re-encoding at all, so both
        # the original and its "duplicate" are literally the same bytes on disk. This is
        # the ONLY case exact-hash matching (MD5) can ever catch, by design.
        ext = Path(r.source_path).suffix or '.jpg'
        out_path = dest_dir / f'{r.image_id}{ext}'
        shutil.copyfile(r.source_path, out_path)
    else:
        src_img = Image.open(r.source_path).convert('RGB')
        out_path = dest_dir / f'{r.image_id}.jpg'
        if not r.is_derived:
            out_img, save_kwargs = src_img, {'quality': 92}
        elif r.variant_type == 'near_exact':
            out_img, save_kwargs = make_near_exact(src_img, img_rng)
        else:  # realistic
            out_img, save_kwargs = make_realistic(src_img, img_rng)
        out_img.save(out_path, 'JPEG', **save_kwargs)
    file_paths.append(str(out_path))

gt['file_path'] = file_paths
gt.to_csv(OUT_DIR / 'ground_truth.csv', index=False)
print(f'Wrote {(~gt.is_derived).sum()} images to {ORIG_DIR}')
print(f'Wrote {gt.is_derived.sum()} images to {DUP_DIR}')
print(gt[['image_id', 'is_derived', 'variant_type', 'session_id', 'timestamp']].head(6))


In [ ]:
# ===== Build the all-pairs ground truth table =====
# 100 choose 2 = 4,950 pairs; 20 are true duplicates, 4,930 are true negatives.
n = len(gt)
ii, jj = np.triu_indices(n, k=1)
pairs = pd.DataFrame({
    'i': ii, 'j': jj,
    'image_a': gt.image_id.values[ii], 'image_b': gt.image_id.values[jj],
})
leaf_id = gt.true_leaf_id.values
pairs['true_duplicate'] = (leaf_id[ii] == leaf_id[jj]).astype(int)

# For the 20 true-duplicate pairs, carry through which variant type and time gap produced them.
variant_of = gt.set_index('true_leaf_id').variant_type.to_dict()
gap_of = gt.set_index('true_leaf_id').time_gap_from_source_s.to_dict()
pairs['variant_type'] = [variant_of.get(leaf_id[a]) if leaf_id[a] == leaf_id[b] else None
                          for a, b in zip(ii, jj)]
pairs['time_gap_s'] = [gap_of.get(leaf_id[a]) if leaf_id[a] == leaf_id[b] else None
                        for a, b in zip(ii, jj)]
# variant_type/time_gap were stored on whichever of the pair is_derived; recover from the derived row directly
derived_leaf_to_variant = gt[gt.is_derived].set_index('true_leaf_id')[['variant_type', 'time_gap_from_source_s']]
pairs.loc[pairs.true_duplicate == 1, 'variant_type'] = pairs.loc[pairs.true_duplicate == 1, 'image_a'].map(
    gt.set_index('image_id').true_leaf_id).map(derived_leaf_to_variant['variant_type'])
pairs.loc[pairs.true_duplicate == 1, 'time_gap_s'] = pairs.loc[pairs.true_duplicate == 1, 'image_a'].map(
    gt.set_index('image_id').true_leaf_id).map(derived_leaf_to_variant['time_gap_from_source_s'])

n_true = int(pairs.true_duplicate.sum())
assert n_true == N_DUP_SOURCES, f'expected {N_DUP_SOURCES} true-duplicate pairs, found {n_true}'
print(f'{len(pairs):,} total pairs | {n_true} true duplicates | {len(pairs) - n_true:,} true negatives')
print(pairs[pairs.true_duplicate == 1].variant_type.value_counts())


In [ ]:
# ===== Method 1: exact-hash matching (MD5 of raw file bytes) =====
def md5_of(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

gt['md5'] = gt.file_path.apply(md5_of)
md5_map = gt.set_index('image_id').md5.to_dict()
pairs['pred_exact_hash'] = (pairs.image_a.map(md5_map) == pairs.image_b.map(md5_map)).astype(int)
print('Exact-hash flagged pairs:', pairs.pred_exact_hash.sum(),
      '| of which true duplicates:', pairs.loc[pairs.pred_exact_hash == 1, 'true_duplicate'].sum())


In [ ]:
# ===== Compute pHash for every synthetic image (same convention as NB01) =====
t0 = time.time()
gt['phash'] = [imagehash.phash(Image.open(p)) for p in gt.file_path]
print(f'pHash computed for {len(gt)} images in {time.time() - t0:.1f}s')


In [ ]:
# ===== Methods 2 & 3: pHash-only clustering, and the full specimen method =====
# This is the SAME code as the "specimen_id = pHash near-duplicates UNION bounded
# temporal blocks" cell in nb01-identity-disjoint-splits.ipynb, applied here to the
# 100-image synthetic set instead of the 11,094-image real one, so the comparison is a
# faithful replay of the actual paper technique rather than a re-implementation of it.

class UF:
    def __init__(s, n): s.p = list(range(n))
    def find(s, x):
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[rb] = ra

def popcount64(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0f0f0f0f0f0f0f0f)
    return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)

hbits = np.array([int(str(h), 16) for h in gt.phash], dtype=np.uint64)

# --- pHash edges, computed within each class label only (exactly as in NB01) ---
t0 = time.time(); phash_edges = []
for lab, g in gt.groupby('label'):
    idx = g.index.to_numpy(); H = hbits[idx]
    D = popcount64(H[:, None] ^ H[None, :])
    iu, ju = np.triu_indices(len(idx), k=1)
    near = D[iu, ju] <= PHASH_THRESHOLD
    phash_edges.extend(zip(idx[iu[near]], idx[ju[near]]))
print(f'pHash edges: {len(phash_edges)} ({time.time() - t0:.1f}s)')

def temporal_blocks(span_s):
    """BOUNDED blocks: start a new block when the current block's span exceeds span_s.
    Non-transitive by construction, so continuous shooting cannot chain a whole session."""
    blk = np.empty(len(gt), dtype=np.int64); b = 0
    for _, g in gt.groupby('session_id', sort=False):
        gi = g.index.to_numpy(); gtm = g.timestamp.values
        o = np.argsort(gtm, kind='stable'); gi, gtm = gi[o], gtm[o]
        b += 1; start = gtm[0]
        for k in range(len(gi)):
            if gtm[k] - start > span_s:
                b += 1; start = gtm[k]
            blk[gi[k]] = b
    return blk

def build_specimens(span_s):
    uf = UF(len(gt))
    for i_, j_ in phash_edges: uf.union(int(i_), int(j_))
    if span_s > 0:
        blk = temporal_blocks(span_s)
        o = np.argsort(blk, kind='stable'); bs = blk[o]
        starts = np.flatnonzero(np.r_[True, bs[1:] != bs[:-1]])
        for s_, e_ in zip(starts, np.r_[starts[1:], len(o)]):
            grp = o[s_:e_]
            for x in grp[1:]: uf.union(int(grp[0]), int(x))
    return np.array([uf.find(i_) for i_ in range(len(gt))])

# Method 2: cluster_id = pHash only (span_s = 0, i.e. "what the OLD paper did")
cluster_root = build_specimens(0)
gt['cluster_id'] = [f'c{r:05d}' for r in cluster_root]
print('Method 2 (pHash-only) clusters:', gt.cluster_id.nunique())

# Method 3: specimen_id = pHash UNION bounded 30s temporal blocks (the actual paper technique)
specimen_root = build_specimens(SPECIMEN_BLOCK_SECONDS)
gt['specimen_id'] = [f's{r:05d}' for r in specimen_root]
print(f'Method 3 (full specimen, {SPECIMEN_BLOCK_SECONDS}s) specimens:', gt.specimen_id.nunique())

cluster_map = gt.set_index('image_id').cluster_id.to_dict()
specimen_map = gt.set_index('image_id').specimen_id.to_dict()
pairs['pred_phash_only'] = (pairs.image_a.map(cluster_map) == pairs.image_b.map(cluster_map)).astype(int)
pairs['pred_specimen'] = (pairs.image_a.map(specimen_map) == pairs.image_b.map(specimen_map)).astype(int)
print('pHash-only flagged pairs:', pairs.pred_phash_only.sum(),
      '| full-specimen flagged pairs:', pairs.pred_specimen.sum())


In [ ]:
# ===== Plot style + save_fig helper (defined once, used by every figure below) =====
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.alpha': 0.22, 'grid.linewidth': 0.5, 'axes.axisbelow': True,
    'axes.edgecolor': '#666666', 'axes.linewidth': 0.8,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 10,
    'legend.frameon': False, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')


In [ ]:
# ===== Method 4a: ImageNet-pretrained ResNet-50 embeddings (NOT fine-tuned), same as NB08 =====
IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths; self.t = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.t(img), i

backbone = torchvision.models.resnet50(weights='IMAGENET1K_V2')
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()
for p in backbone.parameters():
    p.requires_grad_(False)

paths = gt.file_path.tolist()
loader = DataLoader(ImagePathDataset(paths, eval_tf), batch_size=32, shuffle=False, num_workers=2)

embeddings = np.zeros((len(paths), 2048), dtype=np.float32)
t0 = time.time()
with torch.no_grad():
    for x, idx in loader:
        feats = backbone(x.to(device)).cpu().numpy()
        embeddings[idx.numpy()] = feats
print(f'Embeddings extracted for {len(paths)} images in {time.time() - t0:.1f}s')


In [ ]:
# ===== Method 4b: pairwise cosine similarity + ROC / PR curves =====
norm_emb = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
cos_all = norm_emb @ norm_emb.T
pairs['cosine_sim'] = cos_all[pairs.i.values, pairs.j.values]

y_true = pairs.true_duplicate.values
y_score = pairs.cosine_sim.values

roc_auc = roc_auc_score(y_true, y_score)
pr_auc = average_precision_score(y_true, y_score)
fpr, tpr, roc_thresh = roc_curve(y_true, y_score)
prec_curve, rec_curve, pr_thresh = precision_recall_curve(y_true, y_score)

# F1-optimal operating threshold, chosen WITH ground truth (only possible because this is
# a synthetic benchmark) -- this is the fairest possible reading of cosine similarity,
# in contrast to the percentile-based guess NB08 had to make on the real, unlabeled data.
f1_curve = np.where((prec_curve + rec_curve) > 0,
                     2 * prec_curve * rec_curve / (prec_curve + rec_curve + 1e-12), 0)
best_i = np.argmax(f1_curve[:-1]) if len(f1_curve) > 1 else 0
cosine_best_threshold = float(pr_thresh[best_i]) if len(pr_thresh) else 0.5
pairs['pred_cosine_at_best_f1'] = (pairs.cosine_sim >= cosine_best_threshold).astype(int)

print(f'Cosine similarity: ROC-AUC={roc_auc:.4f}  PR-AUC={pr_auc:.4f}  '
      f'best-F1 threshold={cosine_best_threshold:.4f}')


In [ ]:
# ===== Evaluate every method: precision, recall, F1 against ground truth (PAIRWISE) =====
# Kept for transparency and as an input to the cross-check in the specimen-level metrics
# section below, but pairwise precision is NOT the headline comparison for the paper --
# see the note in the specimen-purity cell for why it unfairly penalises over-merging.
def prf(pred_col):
    tp = int(((pairs[pred_col] == 1) & (pairs.true_duplicate == 1)).sum())
    fp = int(((pairs[pred_col] == 1) & (pairs.true_duplicate == 0)).sum())
    fn = int(((pairs[pred_col] == 0) & (pairs.true_duplicate == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else float('nan')
    recall = tp / (tp + fn) if (tp + fn) else float('nan')
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else float('nan')
    return dict(TP=tp, FP=fp, FN=fn, precision=precision, recall=recall, f1=f1)

rows = []
rows.append({'method': 'Exact-hash (MD5)', **prf('pred_exact_hash'), 'roc_auc': np.nan, 'pr_auc': np.nan})
rows.append({'method': 'pHash-only (old method)', **prf('pred_phash_only'), 'roc_auc': np.nan, 'pr_auc': np.nan})
rows.append({'method': f'Full specimen (ours, {SPECIMEN_BLOCK_SECONDS}s)', **prf('pred_specimen'),
             'roc_auc': np.nan, 'pr_auc': np.nan})
cos_row = prf('pred_cosine_at_best_f1')
rows.append({'method': 'Cosine similarity (ResNet-50)', **cos_row, 'roc_auc': roc_auc, 'pr_auc': pr_auc})

metrics = pd.DataFrame(rows)
metrics.to_csv(OUT_DIR / 'dedup_benchmark_metrics.csv', index=False)
print('Pairwise metrics (reference only -- see specimen-level metrics below for the headline comparison):')
print(metrics.to_string(index=False))


In [ ]:
# ===== Cosine similarity as connected components (for a fair, cluster-level comparison) =====
# Treat every pair with cosine_sim >= the F1-optimal threshold as an edge, then take
# connected components -- exactly the same "graph -> Union-Find -> group" pattern used
# for pHash-only and the full specimen method (the UF class from that earlier cell is
# reused here), so all three can be compared in the same units (specimens/clusters), not
# raw pairwise flags.
uf_cos = UF(len(gt))
cos_edges = pairs.loc[pairs.cosine_sim >= cosine_best_threshold, ['i', 'j']].to_numpy()
for i_, j_ in cos_edges:
    uf_cos.union(int(i_), int(j_))
gt['cosine_cluster_id'] = [f'x{uf_cos.find(i_):05d}' for i_ in range(len(gt))]
gt['exact_hash_group_id'] = gt['md5']  # give the exact-hash grouping a clear name too
print('Cosine-similarity connected components:', gt.cosine_cluster_id.nunique(),
      f'(threshold={cosine_best_threshold:.4f})')


In [ ]:
# ===== Save every image's group assignment under all four methods =====
assignments = gt[['image_id', 'true_leaf_id', 'is_derived', 'variant_type', 'label',
                   'session_id', 'exact_hash_group_id', 'cluster_id', 'specimen_id',
                   'cosine_cluster_id']].copy()
assignments.to_csv(OUT_DIR / 'cluster_assignments.csv', index=False)
print('Saved per-image group assignments for all 4 methods to cluster_assignments.csv')
print(assignments.head(5))


In [ ]:
# ===== Specimen-level purity: a fairer way to score over-merging than pairwise precision =====
# Pairwise precision punishes one over-merge mistake once for EVERY pair inside the bad
# group (a group of 5 wrongly-merged images produces 10 "false positive pairs" from a
# single mistake). That combinatorial blow-up doesn't match how the mistake actually costs
# the paper: one wrongly-merged leaf is one lost independent unit, not ten. This cell
# scores every method in units that match Table 4 in the manuscript (specimen counts),
# not pairs -- and is the comparison that actually matches what specimen construction is
# for: keeping true duplicates together (leakage-prevention recall) while disturbing as
# few genuinely-different leaves as possible (over-merge rate).
N_REAL_LEAVES = N_SINGLETON + N_DUP_SOURCES  # 79 distinct real photos in this benchmark

group_cols = {
    'Exact-hash (MD5)': 'exact_hash_group_id',
    'pHash-only (old method)': 'cluster_id',
    f'Full specimen (ours, {SPECIMEN_BLOCK_SECONDS}s)': 'specimen_id',
    'Cosine similarity (ResNet-50)': 'cosine_cluster_id',
}

def purity_stats(col):
    groups = gt.groupby(col)['true_leaf_id'].apply(lambda s: s.nunique())
    n_groups = int(len(groups))
    impure_groups = groups[groups > 1]
    n_impure = int(len(impure_groups))
    # Leaves entangled with a DIFFERENT leaf's image somewhere in the same group -- counted
    # once per leaf, not once per pair.
    impure_group_ids = impure_groups.index
    leaves_affected = int(gt.loc[gt[col].isin(impure_group_ids), 'true_leaf_id'].nunique())
    over_merge_rate = leaves_affected / N_REAL_LEAVES
    # Leakage-prevention recall, recomputed directly from the grouping as a cross-check
    # against the pairwise recall computed earlier -- the two must agree exactly.
    dup_leaf_ids = gt.loc[gt.is_derived, 'true_leaf_id'].unique()
    caught = sum(1 for lid in dup_leaf_ids if gt.loc[gt.true_leaf_id == lid, col].nunique() == 1)
    leakage_recall = caught / len(dup_leaf_ids)
    return dict(n_groups=n_groups, impure_groups=n_impure, leaves_affected=leaves_affected,
                over_merge_rate=over_merge_rate, leakage_recall=leakage_recall)

purity_rows = [{'method': name, **purity_stats(col)} for name, col in group_cols.items()]
purity = pd.DataFrame(purity_rows)
purity.to_csv(OUT_DIR / 'specimen_purity_metrics.csv', index=False)
print(purity.to_string(index=False))

print('\nCross-check: leakage_recall above should exactly match the earlier pairwise recall:')
print(metrics[['method', 'recall']].rename(columns={'recall': 'pairwise_recall'}))
mismatch = [(m, purity.loc[purity.method == m, 'leakage_recall'].iloc[0], metrics.loc[metrics.method == m, 'recall'].iloc[0])
            for m in purity.method if abs(purity.loc[purity.method == m, 'leakage_recall'].iloc[0]
                                           - metrics.loc[metrics.method == m, 'recall'].iloc[0]) > 1e-9]
assert not mismatch, f'leakage_recall disagrees with pairwise recall: {mismatch}'
print('OK - the two recall computations agree exactly, as they must.')


In [ ]:
# ===== LaTeX table: the fairer, specimen-level comparison (this is the one for the paper) =====
def _esc(s):
    s = str(s)
    for a, b in [('\\', r'\textbackslash{}'), ('_', r'\_'), ('%', r'\%'), ('&', r'\&'),
                 ('#', r'\#'), ('$', r'\$'), ('{', r'\{'), ('}', r'\}')]:
        s = s.replace(a, b)
    return s

def _pct(x):
    return f'{100 * x:.1f}\\%'

lines = [
    "\\begin{table}[H]",
    "\\centering",
    "\\caption{Deduplication methods scored at the specimen level against the 100-image "
    f"ground-truth benchmark ({N_REAL_LEAVES} real leaves, {N_DUP_SOURCES} with a synthetic "
    "duplicate). Leakage-prevention recall is the fraction of true duplicate pairs placed in "
    "the same group -- the quantity that matters for split integrity. Leaves over-merged "
    "counts each wrongly-grouped real leaf once, rather than once per pair, avoiding the "
    "combinatorial inflation pairwise precision gives a single over-merge mistake.}",
    "\\label{tab:dedup_purity}",
    "\\begin{tabular}{lrrrr}",
    "\\toprule",
    "Method & Groups & Impure groups & Leaves over-merged & Leakage recall \\\\",
    "\\midrule",
]
for _, r in purity.iterrows():
    lines.append(f"{_esc(r.method)} & {int(r.n_groups)} & {int(r.impure_groups)} & "
                 f"{int(r.leaves_affected)}/{N_REAL_LEAVES} ({_pct(r.over_merge_rate)}) & "
                 f"{_pct(r.leakage_recall)} \\\\")
lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}", ""]

purity_table_tex = "\n".join(lines)
(OUT_DIR / 'table_dedup_purity.tex').write_text(purity_table_tex)
print(purity_table_tex)


In [ ]:
# ===== Figure: leakage-prevention recall vs. over-merging cost -- the comparison that
# ===== actually matches what specimen construction is for =====
fig, ax = plt.subplots(figsize=(6.5, 5))
colors = {'Exact-hash (MD5)': '#999999', 'pHash-only (old method)': '#DD8452',
          f'Full specimen (ours, {SPECIMEN_BLOCK_SECONDS}s)': '#55A868',
          'Cosine similarity (ResNet-50)': '#4C72B0'}
markers = {'Exact-hash (MD5)': 'o', 'pHash-only (old method)': 's',
           f'Full specimen (ours, {SPECIMEN_BLOCK_SECONDS}s)': '^',
           'Cosine similarity (ResNet-50)': 'D'}
for _, r in purity.iterrows():
    ax.scatter([r.leakage_recall], [r.over_merge_rate], s=150, color=colors[r.method],
               marker=markers[r.method], edgecolor='black', linewidth=0.7, zorder=5,
               label=f'{r.method} ({int(r.leaves_affected)}/{N_REAL_LEAVES} leaves over-merged)')

ax.set_xlabel('Leakage-prevention recall (fraction of true duplicates caught)')
ax.set_ylabel(f'Over-merge rate (fraction of the {N_REAL_LEAVES} real leaves\nwrongly grouped with a different leaf)')
ax.set_title('The trade-off that actually matters for split integrity\n(top-left is best: catch duplicates, disturb nothing else)')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.01, max(0.15, purity.over_merge_rate.max() * 1.3))
ax.legend(fontsize=8, loc='upper left', markerscale=0.55, labelspacing=1.1,
          handletextpad=0.8, borderpad=0.7)
fig.tight_layout()
save_fig(fig, 'fig_dedup_recall_vs_overmerge')
plt.show()


In [ ]:
# ===== Figure: recall by duplicate type (exact-copy, near-exact, realistic) =====
true_pairs_only = pairs[pairs.true_duplicate == 1].copy()
method_cols = {
    'Exact-hash': 'pred_exact_hash',
    'pHash-only': 'pred_phash_only',
    'Full specimen (ours)': 'pred_specimen',
    'Cosine sim.': 'pred_cosine_at_best_f1',
}
variant_types = ['exact_copy', 'near_exact', 'realistic']
breakdown_rows = []
for name, col in method_cols.items():
    for vtype in variant_types:
        sub = true_pairs_only[true_pairs_only.variant_type == vtype]
        recall_v = sub[col].mean() if len(sub) else float('nan')
        breakdown_rows.append({'method': name, 'variant_type': vtype, 'recall': recall_v, 'n': len(sub)})
breakdown = pd.DataFrame(breakdown_rows)
breakdown.to_csv(OUT_DIR / 'recall_by_variant_type.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
methods_list = list(method_cols.keys())
x = np.arange(len(methods_list)); width = 0.25
colors = {'exact_copy': '#999999', 'near_exact': '#8FAFD4', 'realistic': '#E29497'}
labels = {'exact_copy': f'Exact-copy (n={N_EXACT_COPY})', 'near_exact': f'Near-exact / burst (n={N_NEAR_EXACT})',
          'realistic': f'Realistic multi-angle (n={N_REALISTIC})'}
for k, vtype in enumerate(variant_types):
    vals = [breakdown[(breakdown.method == m) & (breakdown.variant_type == vtype)].recall.iloc[0] for m in methods_list]
    ax.bar(x + (k - 1) * width, vals, width, label=labels[vtype], color=colors[vtype])
ax.set_xticks(x); ax.set_xticklabels(methods_list, rotation=12, ha='right')
ax.set_ylabel('Recall (fraction of true duplicate pairs found)')
ax.set_ylim(0, 1.12)
ax.set_title('Which duplicate type does each method actually catch?')
ax.legend(fontsize=7.5)
fig.tight_layout()
save_fig(fig, 'fig_dedup_recall_by_type')
plt.show()
print(breakdown)


In [ ]:
# ===== Time-gap analysis: does the 30 s window actually matter here? =====
# The 20 true-duplicate pairs were generated with gaps spread from 2 to 90 seconds, deliberately
# straddling the paper's chosen 30 s block boundary. This checks, on ground truth, what the
# full specimen method gains (or loses) relative to pHash-only at that specific cutoff.
true_pairs_only['within_30s'] = true_pairs_only.time_gap_s <= SPECIMEN_BLOCK_SECONDS

gap_rows = []
for within in [True, False]:
    sub = true_pairs_only[true_pairs_only.within_30s == within]
    if len(sub) == 0:
        continue
    gap_rows.append({
        'group': f'gap <= {SPECIMEN_BLOCK_SECONDS}s' if within else f'gap > {SPECIMEN_BLOCK_SECONDS}s',
        'n_pairs': len(sub),
        'recall_phash_only': sub.pred_phash_only.mean(),
        'recall_full_specimen': sub.pred_specimen.mean(),
        'recall_cosine': sub.pred_cosine_at_best_f1.mean(),
    })
gap_table = pd.DataFrame(gap_rows)
gap_table.to_csv(OUT_DIR / 'recall_by_time_gap.csv', index=False)
print(gap_table.to_string(index=False))


In [ ]:
# ===== LaTeX comparison table (PAIRWISE, supplementary -- see table_dedup_purity.tex for the headline table) =====
def latex_escape(s):
    s = str(s)
    for a, b in [('\\', r'\textbackslash{}'), ('_', r'\_'), ('%', r'\%'), ('&', r'\&'),
                 ('#', r'\#'), ('$', r'\$'), ('{', r'\{'), ('}', r'\}')]:
        s = s.replace(a, b)
    return s

def fmt(x):
    return '--' if pd.isna(x) else f'{x:.3f}'

n_neg = len(pairs) - N_DUP_SOURCES
lines = [
    "\\begin{table}[H]",
    "\\centering",
    "\\caption{Deduplication methods scored against a 100-image synthetic benchmark with "
    f"known ground truth: {N_DUP_SOURCES} true duplicate pairs ({N_EXACT_COPY} exact byte-copy, "
    f"{N_NEAR_EXACT} near-exact burst-style, {N_REALISTIC} realistic multi-angle) against "
    f"{n_neg:,} true negatives. ROC-AUC/PR-AUC apply only to the "
    "continuous-score cosine-similarity method; the other three are binary decision rules "
    "with no threshold to sweep.}",
    "\\label{tab:dedup_benchmark}",
    "\\begin{tabular}{lrrrrrr}",
    "\\toprule",
    "Method & TP & FP & FN & Precision & Recall & F1 \\\\",
    "\\midrule",
]
for _, r in metrics.iterrows():
    lines.append(f"{latex_escape(r.method)} & {int(r.TP)} & {int(r.FP)} & {int(r.FN)} & "
                 f"{fmt(r.precision)} & {fmt(r.recall)} & {fmt(r.f1)} \\\\")
lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}", ""]

table_tex = "\n".join(lines)
(OUT_DIR / 'table_dedup_benchmark.tex').write_text(table_tex)
print(table_tex)


In [ ]:
# ===== Figure: precision-recall curve (PAIRWISE, supplementary -- see fig_dedup_recall_vs_overmerge for the headline figure) =====
def _fmt_pr(x):
    return 'n/a' if pd.isna(x) else f'{x:.2f}'

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rec_curve, prec_curve, color='#4C72B0', linewidth=1.6,
        label=f'Cosine similarity (PR-AUC={pr_auc:.3f})')

marker_specs = [
    ('Exact-hash', metrics.loc[metrics.method == 'Exact-hash (MD5)'].iloc[0], '#999999', 'o'),
    ('pHash-only', metrics.loc[metrics.method == 'pHash-only (old method)'].iloc[0], '#DD8452', 's'),
    ('Full specimen (ours)', metrics.loc[metrics.method.str.startswith('Full specimen')].iloc[0], '#55A868', '^'),
]
for label, row, color, marker in marker_specs:
    ax.scatter([row.recall], [row.precision], color=color, marker=marker, s=90, zorder=5,
               edgecolor='black', linewidth=0.6,
               label=f'{label} (P={_fmt_pr(row.precision)}, R={_fmt_pr(row.recall)})')

ax.set_xlabel('Recall (fraction of true duplicates found)')
ax.set_ylabel('Precision (fraction of flags that are correct)')
ax.set_title('Deduplication methods vs. ground truth\n(100-image synthetic benchmark)')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=7.5, loc='lower left')
fig.tight_layout()
save_fig(fig, 'fig_dedup_pr_curve')
plt.show()


In [ ]:
# ===== Contact sheet: example true-duplicate pairs and which methods caught them =====
img_a_map = gt.set_index('image_id').file_path.to_dict()
example_pool = true_pairs_only.sort_values(['variant_type', 'time_gap_s']).reset_index(drop=True)
K = min(6, len(example_pool))
examples = example_pool.iloc[np.linspace(0, len(example_pool) - 1, K).astype(int)]

fig, axes = plt.subplots(K, 2, figsize=(4.5, 2.3 * K))
if K == 1:
    axes = axes.reshape(1, 2)
for row_i, (_, r) in enumerate(examples.iterrows()):
    for col_i, img_id in enumerate([r.image_a, r.image_b]):
        ax = axes[row_i, col_i]
        ax.imshow(Image.open(img_a_map[img_id]))
        ax.axis('off')
    caught_by = [name for name, col in [('exact', 'pred_exact_hash'), ('pHash', 'pred_phash_only'),
                                         ('specimen', 'pred_specimen'), ('cosine', 'pred_cosine_at_best_f1')]
                 if r[col] == 1]
    axes[row_i, 0].set_title(f'{r.variant_type}, gap={int(r.time_gap_s)}s\ncaught by: {", ".join(caught_by) or "none"}',
                              fontsize=7.5, loc='left')
plt.tight_layout()
save_fig(fig, 'fig_dedup_contact_sheet')
plt.show()


In [ ]:
# ===== Final manuscript-numbers summary (JSON + markdown) =====
summary = {
    'benchmark': {
        'n_images': int(N_TOTAL_IMAGES),
        'n_real_leaves': int(N_REAL_LEAVES),
        'n_singleton': int(N_SINGLETON),
        'n_true_duplicate_pairs': int(N_DUP_SOURCES),
        'n_exact_copy_pairs': int(N_EXACT_COPY),
        'n_near_exact_pairs': int(N_NEAR_EXACT),
        'n_realistic_pairs': int(N_REALISTIC),
        'n_true_negative_pairs': int(len(pairs) - N_DUP_SOURCES),
        'phash_threshold': int(PHASH_THRESHOLD),
        'specimen_block_seconds': int(SPECIMEN_BLOCK_SECONDS),
    },
    'specimen_level_metrics_HEADLINE': purity.to_dict(orient='records'),
    'pairwise_metrics_reference_only': metrics.to_dict(orient='records'),
    'recall_by_variant_type': breakdown.to_dict(orient='records'),
    'recall_by_time_gap': gap_table.to_dict(orient='records'),
    'cosine_best_f1_threshold': float(cosine_best_threshold),
}
json.dump(summary, open(OUT_DIR / 'dedup_benchmark_results.json', 'w'), indent=2)

md_lines = ["# NB09 manuscript numbers — ground-truth deduplication benchmark", ""]
md_lines.append(f"Synthetic benchmark: {N_TOTAL_IMAGES} images ({N_REAL_LEAVES} real leaves in "
                 f"originals/, {N_DUP_SOURCES} synthetic duplicates in duplicates/), "
                 f"{N_DUP_SOURCES} true duplicate pairs ({N_EXACT_COPY} exact-copy, "
                 f"{N_NEAR_EXACT} near-exact / burst-style, {N_REALISTIC} realistic multi-angle), "
                 f"{len(pairs) - N_DUP_SOURCES:,} true negatives.")
md_lines.append("")
md_lines.append("## Headline: specimen-level comparison (use this one)")
md_lines.append("")
md_lines.append("| Method | Groups | Impure groups | Leaves over-merged | Leakage recall |")
md_lines.append("|---|---|---|---|---|")
for _, r in purity.iterrows():
    md_lines.append(f"| {r.method} | {int(r.n_groups)} | {int(r.impure_groups)} | "
                     f"{int(r.leaves_affected)}/{N_REAL_LEAVES} ({100*r.over_merge_rate:.1f}%) | "
                     f"{100*r.leakage_recall:.1f}% |")
md_lines.append("")
md_lines.append("## Reference only: pairwise precision/recall (inflates over-merge mistakes combinatorially)")
md_lines.append("")
md_lines.append("| Method | Precision | Recall | F1 | TP | FP | FN |")
md_lines.append("|---|---|---|---|---|---|---|")
for _, r in metrics.iterrows():
    md_lines.append(f"| {r.method} | {r.precision:.3f} | {r.recall:.3f} | {r.f1:.3f} | "
                     f"{int(r.TP)} | {int(r.FP)} | {int(r.FN)} |")
md_lines.append("")
md_lines.append(f"Cosine similarity: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f}, "
                 f"best-F1 threshold={cosine_best_threshold:.4f} (chosen using ground truth, "
                 "only possible because this benchmark is synthetic -- not reproducible on the "
                 "real, unlabeled 11,094-image dataset).")
md_lines.append("")
md_lines.append("Recall by duplicate type (pairwise):")
for _, r in breakdown.iterrows():
    md_lines.append(f"- {r.method} / {r.variant_type}: {r.recall:.3f} (n={int(r.n)})")
md_lines.append("")
md_lines.append(f"Recall by true-pair time gap relative to the {SPECIMEN_BLOCK_SECONDS}s block boundary:")
for _, r in gap_table.iterrows():
    md_lines.append(f"- {r.group} (n={int(r.n_pairs)}): pHash-only={r.recall_phash_only:.3f}, "
                     f"full specimen={r.recall_full_specimen:.3f}, cosine={r.recall_cosine:.3f}")

(OUT_DIR / 'manuscript_numbers_dedup_benchmark.md').write_text("\n".join(md_lines))
print("\n".join(md_lines))
print("\nAll output files:")
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(OUT_DIR))


## What to bring back

Send back the whole `revision_dedup_benchmark` folder from `/kaggle/working/`. In particular:

* `table_dedup_purity.tex` / `specimen_purity_metrics.csv` — the **headline** comparison
  (leakage-prevention recall + over-merge rate, scored fairly at the specimen level).
* `cluster_assignments.csv` — every image's group under all four methods, if you want to
  inspect exactly which leaves got over-merged.
* `fig_dedup_recall_vs_overmerge.png` — the headline figure.
* `dedup_benchmark_metrics.csv` / `table_dedup_benchmark.tex` / `fig_dedup_pr_curve.png` —
  the older pairwise comparison, kept for reference only.
* `dedup_benchmark_results.json` and `manuscript_numbers_dedup_benchmark.md` — every
  number from both scoring passes in one place.
* `recall_by_variant_type.csv`, `recall_by_time_gap.csv`, and the contact sheet — supporting detail.

Once I have these, I can tell you honestly how the full specimen method actually compares
to plain pHash and cosine similarity on ground truth, in the units that match what
specimen construction is actually for, and fold whichever numbers are strong enough into
the manuscript's Section 3.4 or into a new short validation subsection.
